# Creating a Steganography CLI

[Steganography](https://en.wikipedia.org/wiki/Steganography) encompasses techniques for concealing a message. It differs from cryptography, which aims not to hide a message but to render it incomprehensible to anyone without the necessary key.

The method we will implement in this practical work is well-known: it involves [modifying the least significant bit of each pixel in an image](https://en.wikipedia.org/wiki/Steganography#Least_significant_bit). This modification is not visible to the naked eye (although it can be detected through statistical analysis).

## Work Environment

For the labs, we will use GitHub Codespaces, which provides VSCode instances within containers. The free version allows 30 hours of usage per month with 15GB of storage. This is more than sufficient for this practical work.

If you do not have a GitHub account, start by creating one [here](https://github.com/signup).

[![Access the work environment](https://github.com/codespaces/badge.svg)](https://github.com/codespaces/new?hide_repo_select=true&ref=main&repo=572729501&machine=basicLinux32gb&devcontainer_path=.devcontainer%2Fdevcontainer.json&location=WestEurope)

## Step 1 ⋅ Functions for Converting Strings to Bits and Vice Versa

First, we will focus on converting strings into bits (and vice versa).

A first function, `encode_string`, will take a string as an argument and return a numpy array of the corresponding bits (in utf8 encoding).

A numpy array is a homogeneous data structure (elements of the same type), which we will use in this practical work as a highly enhanced list.

For example, for the string `"ABC"`, we expect the following output array:

```python
numpy.array([0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1])
```

Indeed, `A` is encoded as `01000001` in utf8, `B` as `01000010`, and `C` as `01000011`. The expected array is simply these bits concatenated.

To code this function, here are some useful functions:
- [`str.encode`](https://docs.python.org/3/library/stdtypes.html#str.encode) which returns a `bytes` object (itself an iterable of bytes represented as integers)
- [`numpy.unpackbits`](https://numpy.org/doc/stable/reference/generated/numpy.unpackbits.html), which you can apply to a numpy array previously created from the bytes represented as integers obtained from the `bytes` object. To create the numpy array, you can use [`numpy.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) or [`numpy.fromiter`](https://numpy.org/doc/stable/reference/generated/numpy.fromiter.html) and specify `dtype="uint8"` during the call. This argument will allow you to represent the byte values with the most suitable type.

Here is the expected function signature:

```python
def encode_string(string: str) -> numpy.ndarray:
```

The `decode_bits` function will handle the reverse transformation. The functions to use are the inverses of those needed for `encode_string`:
- [`bytes.decode`](https://docs.python.org/3/library/stdtypes.html#bytes.decode) which returns a string
- [`numpy.packbits`](https://numpy.org/doc/stable/reference/generated/numpy.packbits.html)

Here is the expected function signature:

```python
def decode_bits(bits: numpy.ndarray) -> str:
```

Instructions:

- Create the file `tests/test_string_conversion.py` and create at least one test case per function, using the given example if necessary. To compare two numpy arrays, you can use the dedicated function [`numpy.testing.assert_array_equal`](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_array_equal.html).
- Create the file `steganosaure/string_conversion.py` and implement these two functions inside so that the tests pass.
- Can you design a test that ensures the two functions are indeed inverses of each other?

## Step 2 ⋅ Functions for Loading and Saving Images

Now we are going to code two important functions: one will load an image, and the other will save it.

**Note, for our algorithm to work, an image saved and then reloaded must be identical to the original image.**

The loading function will need to load an image into a numpy array with elements as 8-bit integers (for this, you can use the [`scikit-image`](https://scikit-image.org/) package and specifically the functions [`skimage.io.imread`](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imread) and [`skimage.util.img_as_ubyte`](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.img_as_ubyte)). Here is the expected signature:

```python
def load_image(image_path: pathlib.Path) -> numpy.ndarray:
```

The saving function will need to save a numpy array to the specified path. Note that to ensure the reloaded image is equivalent to the saved image while maintaining good quality, it is preferable to save images in `.png` format. You can use the function [`skimage.io.imsave`](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imsave). It will be useful to use the arguments `plugin="pil"` and `check_contrast=False`. Here is the expected signature:

```python
def save_image(image: numpy.ndarray, image_path: pathlib.Path) -> None:
```

Instructions:

- Create the file `tests/test_image_io.py` and create at least one test case to ensure that the crucial point is respected (an image saved and then loaded is the same as the original image).
- Create the file `steganosaure/image_io.py` and implement these two functions inside so that the test(s) pass.

## Step 3 ⋅ Bit Manipulation Functions

We will now implement the core of the algorithm: modifying the least significant bit of each pixel to store a bit of information from the string to be hidden.

A brief reminder on bit manipulation:

```python
0b01 & 0b11 == 0b01  # bitwise AND, result bit is 1 if both bits are 1
0b01 | 0b11 == 0b11  # bitwise OR, result bit is 1 if at least one bit is 1
0b11 >> 1 == 0b01    # right shift, fills with 0s
0b11 << 1 == 0b10    # left shift, fills with 0s
```

These representations are equivalent to their decimal notation:

```python
1 & 3 == 1
1 | 3 == 3
3 >> 1 == 1
3 << 1 == 2
```

To achieve this, we will implement two functions. The first one, `retrieve_least_significant_bits`, will return a numpy array of the same size as its argument but containing only 0s or 1s: the least significant bits of the numbers contained in the argument array.

To write this function, feel free to utilize the fact that you can directly apply an operation on a numpy array (including bitwise operations):

```python
>>> 3 & 1
1
>>> 2 & 1
0
>>> numpy.array([3, 2]) & 1
array([1, 0])
```

Here is its expected signature:

```python
def retrieve_least_significant_bits(array: numpy.ndarray) -> numpy.ndarray:
```

The second function to implement, `modify_least_significant_bits`, will modify the least significant bits of the `image` so that they match the `message` array. Here is its expected signature:

```python
def modify_least_significant_bits(message: numpy.ndarray, image: numpy.ndarray) -> numpy.ndarray:
```

Instructions:

- Create the file `tests/test_bits_manipulation.py` and create at least one test case for each function.
- Create the file `steganosaure/bits_manipulation.py` and implement these two functions inside so that the tests pass.

## Step 4 ⋅ Functions to Manipulate numpy Arrays

The third function to implement is `pad_array`. Its purpose is to preprocess the arguments for `modify_least_significant_bits` so that they are of the same size. This function should return a `small_array` (which will be our message to hide) of the same size as the `big_array` (which will be the image where the message is concealed). To achieve this, `small_array` will be padded with zeros. You can use [`numpy.pad`](https://numpy.org/doc/stable/reference/generated/numpy.pad.html) for implementation.

Here is its expected signature:

```python
def pad_array(small_array: numpy.ndarray, big_array: numpy.ndarray) -> numpy.ndarray:
```

## Step 5 ⋅ Implementation of Steganography

We have now completed the preparatory phase and have all the necessary functions to implement steganography.

The first function of our tool, `encrypt`, will allow the user to hide a message within an image. Before applying the already developed methods, it will be necessary to use [`numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) to obtain all pixel values in a one-dimensional array.

Here is its expected signature:

```python
def encrypt(message: str, input_image_path: pathlib.Path, output_image_path: pathlib.Path) -> None:
```

The second function of our tool, `decrypt`, will allow the user to retrieve the previously hidden message from an image. Here is its expected signature:

```python
def decrypt(image_path: pathlib.Path) -> str:
```

Instructions:

- Create the file `tests/test_steganography.py` and write at least one test case to verify the correct functioning of both functions together.
- Create the file `steganosaure/steganography.py` and implement these two functions inside to ensure the tests pass.

### Step 6 ⋅ Creating an Entry Point

For this step, we will create a CLI entry point for our `steganosaure` project.

1. **Creating the `cli.py` File**: Create this file in your source directory (`steganosaure`).

2. **Implementing the `main` Function**: In this file, write a `main` function that takes no arguments and simply prints `Hello World`.

3. **Modifying the `pyproject.toml` File**: Edit this file to configure the `steganosaure` command to execute the `main` function from `cli.py`. Add the following line under `[tool.poetry.scripts]`:

   ```ini
   steganosaure = 'steganosaure.cli:main'
   ```

   Make sure to replace `steganosaure` with your project's name and `cli` with the filename where the `main` function resides.

4. **Installing Dependencies**: Run `poetry install` to apply the changes to the entry point and prepare the project for use.

5. **Testing the Entry Point**: Test the entry point by executing the `steganosaure` command. It should display `Hello World` if everything is configured correctly.

Instructions:

- Ensure that you follow all steps in order to ensure the proper functioning of the CLI entry point.
- You can customize the `main` function to perform specific actions related to your project once the entry point configuration is successful.

In [ ]:
!poetry install

In [ ]:
!steganosaure

### Step 7 ⋅ Adding a Parser

In this step, we will modify the `main` function in the `cli.py` file to handle a command (`encrypt` or `decrypt`) along with the required arguments:

- For `encrypt`: `message`, `input_image_path`, and `output_image_path`
- For `decrypt`: `image_path`

We will draw inspiration from the example slides for this step, as it closely aligns with what we aim to achieve here.

Once these arguments are parsed, we will connect them to the corresponding functions.

Instructions:

- Modify the `main` function in the `cli.py` file to include handling of command-line arguments and parsing.
- Use a parser library such as `argparse` or `click` to facilitate command and argument management.
- Connect the parsed arguments to the `encrypt` and `decrypt` functions already implemented in your project.

## Solution

[Labs Repository](https://github.com/mlambda/tp-steganosaure/tree/solution).